In [ ]:
# Real-Data Fairness Experiment
Test conditional independence $R \perp Z \mid X$ on real-world datasets via **CRT + cGAN**.

Supported datasets: `adults`, `propublica`, `law`, `edu`.  
Change `dataset_name` below to switch.

In [ ]:
import sys, os

# Add src/ to path (works from any location as long as notebook is in the repo root)
REPO_DIR = os.path.dirname(os.path.abspath("__file__"))
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from sklearn.model_selection import KFold
from tqdm.notebook import tqdm

from utils import load_dataset, provide_x_z, convert_scores_to_ranks
from crt_cgan import ConditionalGAN, crt_calibration_efficient
from methods import (Kcondor_v2, nkci_score, nhsic, partial_corr_pg_score)

In [ ]:
## Configuration

In [ ]:
# ── Choose dataset ──────────────────────────────────────────
# Options: "adults", "propublica", "law", "edu"
dataset_name = "adults"

# ── Experiment hyper-parameters ────────────────────────────
K = 5           # number of CV folds for cGAN
B = 200         # bootstrap iterations for CRT
s_size = 1000   # sample size per class (balanced)

# ── Methods to evaluate ────────────────────────────────────
methods = {
    "KCondor":       Kcondor_v2,
    "nKCI":          nkci_score,
    "nHSIC":         nhsic,
    "Partial Corr.": partial_corr_pg_score,
}

In [ ]:
## Load & preprocess data

In [ ]:
df, y_name, features_Z = load_dataset(dataset_name)
print(f"Target: {y_name}")
print(f"Protected feature groups: {features_Z}")
df.head()

In [ ]:
print(f"Class distribution ({y_name}):")
print(df[y_name].value_counts())

In [ ]:
## Run CRT experiment

In [ ]:
preds_o = df[y_name].values          # use the true label as the ranking score
results_all = {}

for i, f_p in enumerate(features_Z):
    print(f"\n{'='*60}")
    print(f"Feature group {i+1}/{len(features_Z)}: {f_p}")
    print(f"{'='*60}")

    # Prepare X, Z splits
    X_np, Z_np, y, X_np_all, Z_np_all, y_all = provide_x_z(
        df, y_name=y_name, f_p=f_p, sample_size_per_class=s_size, fz=features_Z
    )

    # Train K cGAN generators (K-fold)
    kf = KFold(n_splits=K, shuffle=True, random_state=42)
    kf_splits = list(kf.split(X_np))
    print(f"Training {K} cGAN generators ...")
    trained_generators = []
    for train_idx, _ in tqdm(kf_splits, desc="Training generators"):
        generator = ConditionalGAN(x_dim=X_np_all.shape[1], z_dim=Z_np_all.shape[1])
        generator.fit(X_np_all[train_idx], Z_np_all[train_idx])
        trained_generators.append(generator)

    # Compute p-values for every method
    results = {name: [] for name in methods}
    for name, func in tqdm(methods.items(), desc="Methods"):
        p_val = crt_calibration_efficient(
            X_np, Z_np, preds_o,
            scoring_function=func,
            kf_splits=kf_splits,
            trained_generators=trained_generators,
            B=B,
        )
        results[name].append(p_val)
        print(f"  {name}: p = {p_val:.4f}")

    results_all[i] = results

print("\nDone.")

In [ ]:
results_all

In [ ]:
custom_cmap = LinearSegmentedColormap.from_list(
    "pvalue_cmap",
    [(0.0, "red"), (0.05, "white"), (1.0, "green")]
    )

# Convert results_all to a DataFrame
results_df = pd.DataFrame()
for feature_idx, methods_results in results_all.items():
    feature_name = "+".join(features_Z[feature_idx])  # Join all feature names
    for method_name, p_values in methods_results.items():
        results_df.loc[feature_name, method_name] = p_values[0]  # Extract the p-value

plt.figure(figsize=(10, 6))
sns.heatmap(results_df.T, annot=True, cmap=custom_cmap, fmt=".3f", vmin=0, vmax=1, cbar_kws={'label': 'P-Value'})
plt.title(f'Heatmap P-Values: Metodo vs. Feature - Dataset: {dataset_name}')
plt.show()

In [ ]:
# save results_all to pickle
import pickle
with open(f"exp_real_dataset_name_{dataset_name}_B_{B}_K_{K}_size_{s_size}_drop.pkl", "wb") as f:
    pickle.dump(results_all, f)

In [ ]:
import pickle
from pathlib import Path

# Define datasets configuration (pkl files are in OLD/ archive)
OLD_DIR = Path(REPO_DIR) / "OLD"

datasets_config = {
    "Adults": {
        "pkl": OLD_DIR / "exp_real_dataset_name_adults_B_200_K_5_size_2000.pkl",
        "features_Z": [["gender"], ["race"], ["gender+race"]],
    },
    "Compas": {
        "pkl": OLD_DIR / "exp_real_dataset_name_propublica_B_200_K_5_size_20000.pkl",
        "features_Z": [["African_American"], ["Female"], ["Afr_Am+Female"]],
    },
    "Law School": {
        "pkl": OLD_DIR / "exp_real_dataset_name_law_B_200_K_5_size_20000.pkl",
        "features_Z": [["sex"], ["race_nonwhite"], ["sex+race"]],
    },
    "Student Perf.": {
        "pkl": OLD_DIR / "exp_real_dataset_name_edu_B_200_K_5_size_20000.pkl",
        "features_Z": [["sex"], ["address"], ["sex+address"]],
    },
}

# Load all results
all_results = {}
for ds_name, cfg in datasets_config.items():
    try:
        with open(cfg["pkl"], "rb") as f:
            all_results[ds_name] = pickle.load(f)
        print(f"✓ Loaded {ds_name}")
    except FileNotFoundError:
        print(f"✗ Not found: {cfg[chr(39)+chr(39)]}".replace(chr(39)+chr(39), "pkl"))

print(f"
Datasets loaded: {list(all_results.keys())}")


In [ ]:
# Peek at structure of each loaded result to verify feature names
for ds_name, res in all_results.items():
    cfg = datasets_config[ds_name]
    print(f"\n--- {ds_name} ---")
    for feat_idx, methods_results in res.items():
        feat_name = cfg["features_Z"][feat_idx][0]
        for mname, pvals in methods_results.items():
            print(f"  {feat_name} | {mname}: {pvals}")

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Method display names
method_display = {
    'Condor': 'KCondor',
    'nKCI': 'nKCI',
    'nhsic': 'nHSIC',
    'partial_dcorr_score': 'Partial Corr.',
}

# Feature display names per dataset
feature_display = {
    'gender': 'Gender', 'race': 'Race', 'random': 'Random',
    'African_American': 'Afr. American', 'Female': 'Female',
    'sex': 'Sex', 'race_nonwhite': 'Race', 'address': 'Address',
}

custom_cmap = LinearSegmentedColormap.from_list(
    "pvalue_cmap",
    [(0.0, "red"), (0.051, "white"), (1.0, "green")]
)

ds_names = list(all_results.keys())
n_ds = len(ds_names)
fig, axes = plt.subplots(1, n_ds, figsize=(5.5 * n_ds, 5), sharey=True)

for col, ds_name in enumerate(ds_names):
    ax = axes[col]
    cfg = datasets_config[ds_name]
    res = all_results[ds_name]

    # Build DataFrame: rows = methods, columns = features
    df_hm = pd.DataFrame()
    for feat_idx, methods_results in res.items():
        feat_name = feature_display.get(cfg["features_Z"][feat_idx][0], cfg["features_Z"][feat_idx][0])
        for mname, pvals in methods_results.items():
            df_hm.loc[method_display.get(mname, mname), feat_name] = pvals[0]

    sns.heatmap(df_hm, annot=True, cmap=custom_cmap, fmt=".3f", ax=ax,
                vmin=0, vmax=1, cbar=False,
                linewidths=0.5, linecolor='black',
                annot_kws={"size": 18})

    ax.set_title(ds_name, fontsize=26)
    ax.tick_params(axis='both', which='major', labelsize=20)
    ax.tick_params(axis='x', rotation=30)
    ax.tick_params(axis='y', rotation=0)

    if col == 0:
        ax.set_ylabel('')
    else:
        ax.set_ylabel('')

# Colorbar spanning the full height
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('P-Value', fontsize=24)
cbar.ax.tick_params(labelsize=20)

plt.tight_layout(rect=[0, 0, 0.91, 1.0])
plt.show()

fig.savefig("heatmap_multi_dataset_pvalues.pdf", bbox_inches='tight')
print("Saved: heatmap_multi_dataset_pvalues.pdf")